<a href="https://colab.research.google.com/github/CBravoR/AdvancedAnalyticsLabs/blob/master/notebooks/python/Lab_Prompt_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompt engineering with Gemma 4 (E2B, instruction-tuned)

In this lab we explore prompt engineering for large language models (LLMs) using [Gemma 4 E2B-it](https://huggingface.co/google/gemma-4-E2B-it), Google's open-weight model in the "effective 2 billion parameters" size (5.1 billion in total, 2.3 billion active per token). The `-it` suffix means the model is instruction-tuned: it was trained to follow instructions and hold a conversation, which is what we need. It accepts text, images and audio; we use text only.

The model is released under the Apache 2.0 licence which means it can be used freely for both commercial and non-commercial purposes. It runs in bfloat16 on the free Colab GPU (T4 or better). Select a GPU runtime before you start (Runtime → Change runtime type).

We will start by installing and importing the required libraries. Gemma 4 needs a recent version of `transformers`, so the install upgrades it.

In [ ]:
%pip install -U transformers accelerate
%pip install -U shap

In [ ]:
# Imports
import os
import numpy as np
import pandas as pd

# Longer download timeout: the model weights are about 10 GB
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Plots
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# PyTorch and Hugging Face
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, set_seed

# XGBoost and SHAP
from xgboost import XGBClassifier
import shap

# Markdown output
from IPython.display import Markdown, display

## Basic prompt engineering

Gemma 4 is a multimodal model, so Hugging Face exposes it through a **processor** (which handles text, images and audio and holds the chat template) and a multimodal model class. We only feed it text, and the calls are the same as for a text-only model.

Downloading the weights takes a few minutes the first time. `device_map="auto"` places the model on the GPU, and `dtype="auto"` picks the precision the model was published in (bfloat16).

In [ ]:
model_id = "google/gemma-4-E2B-it"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForMultimodalLM.from_pretrained(model_id, dtype="auto", device_map="auto")
model.eval()
print(f"Loaded {model_id} on {model.device} in {model.dtype}")

The following code displays the **chat template** of Gemma 4. A chat template is the recipe that turns a list of messages (with roles such as `system`, `user` and `assistant`) into the single string of tokens the model actually reads. Look for the markers that open and close each turn, and for the place where the system message goes: the model never sees "roles", only this string.

If a model has no chat template, prompts must be formatted by hand. Gemma 4 has one, and it supports a system role natively.

In [ ]:
print(processor.chat_template[:2000])   # the template is long; the first part shows the structure

We wrap generation in one helper so that every experiment below changes only the messages and the decoding parameters. The helper applies the chat template, generates, and decodes **only the new tokens**, so the answer comes back without the prompt.

Gemma 4 can produce a hidden "thinking" section before the answer. The helper strips it if it appears, so that what you read is the reply.

In [ ]:
def generate(messages, max_new_tokens=512, temperature=0.2, top_p=0.8, repetition_penalty=1.5, seed=42):
    """Generate a reply to a list of chat messages and return only the new text."""
    set_seed(seed)
    inputs = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
        )
    new_tokens = out[0, inputs["input_ids"].shape[-1]:]
    text = processor.decode(new_tokens, skip_special_tokens=True)
    # If a thinking channel is present, keep only what follows it.
    if "<channel|>" in text:
        text = text.split("<channel|>")[-1]
    return text.strip()

A prompt is a list of messages. The **system message** sets the model's behaviour (who it is, how it answers); the **user message** is the question. We keep them apart because the system message is the part we will engineer.

In [ ]:
system_message = "You are an expert assistant specializing in Banking Analytics and Business Analytics. Provide structured and factual responses."
user_prompt = "Explain the key differences between Banking Analytics and Business Analytics"

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt},
]
messages

The decoding parameters decide how the next token is chosen from the model's probabilities:

* `max_new_tokens=512` limits the length of the answer.
* `do_sample=True` samples instead of always taking the most likely token.
* `top_p=0.8` (nucleus sampling) restricts sampling to the smallest set of tokens whose probabilities add up to 80%.
* `temperature=0.2` flattens or sharpens the distribution: low values (0.2) give focused, repeatable answers; high values (1.0) give varied, creative ones.
* `repetition_penalty=1.5` discourages repeating the same phrases.

The model card recommends `temperature=1.0`, `top_p=0.95` and `top_k=64` for general chat. A risk analyst wants the opposite: low temperature and a narrow nucleus, so that the same question gives the same answer. We fix the seed for the same reason.

In [ ]:
answer = generate(messages, max_new_tokens=512, temperature=0.2, top_p=0.8, repetition_penalty=1.5)
print(answer)

It works. The answer is Markdown, so we can render it instead of printing it.

In [ ]:
display(Markdown(answer))

The response is well-written, but with prompt engineering, we can refine it further for improved structure and formatting.

Now, let's enhance it by providing more specific instructions for the response.

In [ ]:
system_message = """STRICT INSTRUCTIONS:
1. First, provide a **clear definition** of the topic.
2. Then, explain **at least two key differences** in a **structured manner**.
   - Each difference must be in a **separate paragraph**.
   - Use **clear and concise language**.
3. Include **only** necessary information and useful details that are in line with the requested explanation.
"""

user_prompt = "What is Banking Analytics, and how does it differ from general Business Analytics?"

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt},
]

In [ ]:
answer = generate(messages, max_new_tokens=512, temperature=0.2, top_p=0.8, repetition_penalty=1.5)
display(Markdown(answer))

The model's response is now more narrative and engaging, accurately following the instructions provided.

## Prompt engineering with prediction model

In this section, we will use prompt engineering to guide the LLM in explaining delinquency status predictions based on SHAP values.

Let's begin by downloading a dataset to proceed with our analysis.

In [ ]:
!gdown --fuzzy 'https://drive.google.com/file/d/1nrhxfnAkI0bZRXJiWu_JVKusAD9iHBpK/view?usp=sharing'

In [ ]:
df = pd.read_csv('loan_app.csv')
df

Next, we will prepare the dataset for training by applying feature encoding, scaling, and train-test splitting to ensure the model learns effectively from the data.

In [ ]:
X = df.drop(columns=["target"])  # Features
y = df["target"]  # Target variable

# Convert Categorical Features to Numerical
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Apply One-Hot Encoding
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_encoded = encoder.fit_transform(X[categorical_columns])

# Convert Encoded Data to DataFrame
X_encoded_df = pd.DataFrame(X_encoded, columns=encoder.get_feature_names_out(categorical_columns))

# Drop original categorical columns and merge one-hot encoded features
X = X.drop(columns=categorical_columns)
X = pd.concat([X, X_encoded_df], axis=1)

# Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale Only Numerical Features (NOT one-hot encoded features)
scaler = StandardScaler()
X_train[numerical_columns] = scaler.fit_transform(X_train[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])  # Use same scaler for test set

X_train

Now, we will train an XGBoost model to predict loan delinquency status.

In [ ]:
negative_count = np.sum(y_train == 0)  # Count of class 0
positive_count = np.sum(y_train == 1)  # Count of class 1
scale_pos_weight = negative_count / positive_count  # Weight ratio

# Train XGBoost Classifier
xgb_model = XGBClassifier(max_depth=3,
                          learning_rate=0.01,
                          n_estimators=200,
                          verbosity=0,
                          objective='binary:logistic',
                          eval_metric="logloss",
                          booster='gbtree',
                          n_jobs=-1,
                          gamma=0.001,
                          subsample=0.632,
                          colsample_bytree=1,
                          colsample_bylevel=1,
                          colsample_bynode=1,
                          reg_alpha=0,
                          reg_lambda=0.1,
                          random_state=428,
                          tree_method="hist",
                          scale_pos_weight=scale_pos_weight,
                          # Recent XGBoost versions turn categorical support on by default. Our
                          # features are already one-hot encoded, and SHAP's TreeExplainer refuses
                          # models with the flag on, so we switch it off.
                          enable_categorical=False
                          )

xgb_model.fit(X_train, y_train)

# Make Predictions
y_pred = xgb_model.predict(X_test)

# Evaluate Model Performance
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("Classification Report:\n", classification_report(y_test, y_pred))

We analyzes feature importance in the XGBoost model using SHAP (SHapley Additive exPlanations) to explain why the model predicts a loan as delinquent or not.

The SHAP summary plot provides:
* Feature importance ranking (sorted by impact).
* How each feature affects predictions (positive/negative impact).
* Distribution of SHAP values for different feature values.

In [ ]:
# Explain Model Predictions using SHAP
# The background sample is set on the masker, not on the explainer. TreeSHAP with a
# background ("interventional") runs in time proportional to (background rows) x (explained rows),
# so we explain the first 100 test cases. Explaining all 10,000 takes about 20x longer.
masker = shap.maskers.Independent(X_train, max_samples=100)
explainer = shap.Explainer(xgb_model, masker)
shap_values = explainer(X_test)  # SHAP values for the first 100 test cases

# Summarize SHAP Values
shap.summary_plot(shap_values, X_test)

We retrieve and format SHAP values for one test sample, making it easier to generate human-readable explanations using Gemma.

In [ ]:
# Select an example loan case (e.g., first sample in test set)
sample_index = 10  # Change this index if needed
shap_values_sample = shap_values[sample_index].values
sample_features = X_test.iloc[sample_index]

# Convert SHAP values into dictionary format for input
shap_dict = {feature: shap_values_sample[i] for i, feature in enumerate(sample_features.index)}

# Target label for the selected sample
pred_label = y_pred[sample_index]

print("\nSHAP Values for Sample Client:\n", shap_dict)

Before proceeding to Gemma, let's generate a bar plot of SHAP values for the sampled loan to visualize which features have the most impact on the delinquency prediction.

In [ ]:
shap.plots.bar(shap_values[sample_index])

Finally, we format SHAP values into a structured prompt to enable Gemma to generate an interpretable explanation of why a loan is predicted as delinquent or not.

We first sort SHAP values by absolute magnitude, prioritizing the most impactful features. Then, we select the top 3 features to keep the explanation concise and focused.

In the prompt, we will:

* Ensure a structured format for clear and consistent Gemma responses.
* Eliminate unnecessary or unrelated information from the generated output.
* Clearly state the model's decision using SHAP-based reasoning.

In [ ]:
def format_shap_explanation(system_message, sample_features, shap_dict, pred_label, scaler):
    # Convert scaled numerical values back to original values
    original_values = scaler.inverse_transform(sample_features[numerical_columns].values.reshape(1, -1))
    original_feature_values = {feature: original_values[0][i] for i, feature in enumerate(numerical_columns)}

    # Sort SHAP values by absolute magnitude (most impactful features first)
    top_features = sorted(shap_dict.items(), key=lambda x: abs(x[1]), reverse=True)[:3]

    # Generate SHAP explanation text
    shap_text = "\n".join(
        [f"{feature}: SHAP value = {shap_value:.4f}, feature value = {original_feature_values[feature]:.2f}"
         for feature, shap_value in top_features]
    )

    print(shap_text)
    # Define loan delinquency status
    delinquency_status = "likely to be delinquent" if pred_label == 1 else "unlikely to be delinquent"
    print("predicted delinquency: ", delinquency_status)

    # Construct a revised prompt with explicit instructions and structured format
    user_prompt = f"""
The model predicts that the client is {delinquency_status}.

Here are the three most important features influencing the prediction:

{shap_text}

### Instructions:
- Analyze how each of three features contributes to the prediction.
- **Use correct feature names, not feature values.**
- **Strictly follow the structured response format.**
- **SHAP values must be interpreted correctly**:
  - **A positive SHAP value means the feature increases delinquency risk.**
  - **A negative SHAP value means the feature decreases delinquency risk.**

### Response Format:
- Feature Name: [Feature Name]
- Effect on Risk: [Explain whether a higher or lower value increases delinquency risk and why.]
- SHAP Impact: [Clearly state whether the SHAP value shows an increase or decrease in delinquency risk and explain its significance.]
- Reason: [Provide an individual reason for the source of risk.]

The output will be shown in Markdown format, so style your response accordingly.

### Example Response
**Feature Name: Credit Score**
- Effect on Risk: A higher credit score reduces delinquency risk because it indicates a strong repayment history and financial responsibility.
- SHAP Impact: The SHAP value **-0.5862** shows that **including credit score for this client decreases** the probability of delinquency, meaning the model considers this a strong indicator of financial reliability.
- Reason: A high credit score reflects a history of timely payments and responsible credit usage, which are key indicators of financial stability.

**Feature Name: Employment Length**
- Effect on Risk: A higher employment length reduces delinquency risk because it indicates job stability and a stable salary.
- SHAP Impact: The SHAP value **-0.723** shows that **including employment length for this client decreases** the probability of delinquency, meaning the model considers this a strong indicator of financial reliability. 
- Reason: A long employment length suggests consistent income and financial stability, which reduces the likelihood of delinquency.

Now begin your structured analysis:
"""


    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt},
    ]

    return messages

We also define instructions to guide the LLM in analyzing SHAP values for loan delinquency predictions.

In [ ]:
system_message = """
You are a financial risk analyst. Your job is to analyze SHAP values and provide structured, fact-based explanations of the model's predictions.

Guidelines:
1. Interpret SHAP values correctly:
   - **A positive SHAP value means the feature increases delinquency risk.**
   - **A negative SHAP value means the feature decreases delinquency risk.**
2. **Do not contradict basic financial logic**:
   - A **higher credit score should always reduce risk** unless explicitly stated otherwise.
3. **Strictly follow the response format. Do not add extra text or repeat information.**
4. **Do not argue against the given ranking of features.**
5. **Avoid repetition, unnecessary details, or ranking errors.**
6. **Use clear and concise language.**
7. **Analyze the details of each variable and discuss its individual meaning.**
8. **Provide an unique reply for each variable depending on your beliefs on what may be the source of risk. The space for this will be tagged as _REASON_.
"""

messages = format_shap_explanation(system_message, sample_features, shap_dict, pred_label, scaler)

Finally, let's give it a try!

In [ ]:
answer = generate(messages, max_new_tokens=512, temperature=0.1, top_p=0.5, repetition_penalty=1.0)

In [ ]:
display(Markdown(answer))

Pretty good, no? Of course, a better model will give more detailed and accurate answers. In future labs, we will fine-tune our models to achieve even better performance. Now you can create your own models!